In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from urllib.request import urlretrieve
import numpy as np
from langchain_community.embeddings import HuggingFaceBgeEmbeddings, HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

_= load_dotenv(find_dotenv())

## Document preparation
**We are going to download 4 publications from United States Census Bureau on the following topics:**
- Occupation, Earnings, and Job Characteristics: July 2022
- Household Income in States andMetropolitan Areas: 2022
- Poverty in States and Metropolitan Areas: 2022
- Health Insurance Coverage Status and Type by Geography: 2021 and 2022

We prepare this documents for the LLM to use as a knowledge base.

In [ ]:
# Download documents from U.S. Census Bureau to local directory.
os.makedirs("us_census", exist_ok=True)
files = [
    "https://www.census.gov/content/dam/Census/library/publications/2022/demo/p70-178.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-017.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-016.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-015.pdf",
]
for url in files:
    file_path = os.path.join("us_census", url.rpartition("/")[2])
    urlretrieve(url, file_path)

**Split documents to smaller chunks** 

Documents should be: 
- large enough to contain enough information to answer a question, and 
- small enough to fit into the LLM prompt: Mistral-7B-v0.1 input tokens limited to 4096 tokens
- small enough to fit into the embeddings model: BAAI/bge-small-en-v1.5: input tokens limited to 512 tokens (roughly 2000 characters. Note: 1 token ~ 4 characters).

For this project, we are going to split documents to chunks of roughly 700 characters with an overlap of 50 characters.

In [ ]:
# Load pdf files in the local directory
loader = PyPDFDirectoryLoader("./us_census/")

docs_before_split = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap  = 50,
)
docs_after_split = text_splitter.split_documents(docs_before_split)

In [ ]:
docs_after_split[0]

In [ ]:
avg_doc_length = lambda docs: sum([len(doc.page_content) for doc in docs])//len(docs)
avg_char_before_split = avg_doc_length(docs_before_split)
avg_char_after_split = avg_doc_length(docs_after_split)

print(f'Before split, there were {len(docs_before_split)} documents loaded, with average characters equal to {avg_char_before_split}.')
print(f'After split, there were {len(docs_after_split)} documents (chunks), with average characters equal to {avg_char_after_split} (average chunk length).')

## Text Embeddings with Hugging Face Embedding Models
At the time of writing, there are 213 text embeddings models for English on the [Massive Text Embedding Benchmark (MTEB) leaderboard](https://huggingface.co/spaces/mteb/leaderboard). For our project, we are using LangChain's HuggingFaceBgeEmbeddings (BGE models on the Hugging Face), which according to LangChain are "the best open-source embedding models". Currently, **BAAI/bge-small-en-v1.5** model is the 26th on MTEB leaderboard with max tokens: 512 tokens, embedding dimensions: 384 and model size: 0.13GB.

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
huggingface_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

Now we can see how a sample embedding would look like for one of those chunks.

In [ ]:
sample_embedding = np.array(huggingface_embeddings.embed_query(docs_after_split[0].page_content))
print("Sample embedding of a document chunk: ", sample_embedding)
print("Size of the embedding: ", sample_embedding.shape)

## Retrieval System for vector embeddings
Once we have a embedding model, we are ready to vectorize all our documents and store them in a vector store to construct a retrieval system. With specifically designed searching algorithms, a retrieval system can do similarity searching efficiently to retrieve relevant documents.

FAISS (Facebook AI Similarity Search) is a library that allows developers to quickly search for embeddings of multimedia documents that are similar to each other. It solves limitations of traditional query search engines that are optimized for hash-based searches, and provides more scalable similarity search functions (nearest-neighbor search implementations).

In [ ]:
vectorstore = FAISS.from_documents(docs_after_split, huggingface_embeddings)

In [ ]:
query = """What were the trends in median household income across different states in the United States between 2021 and 2022."""  # Sample question, change to other questions you are interested in.
relevant_documents = vectorstore.similarity_search(query)
print(f'There are {len(relevant_documents)} documents retrieved which are relevant to the query. Display the first one:\n')
print(relevant_documents[0].page_content)

### Create a retriever interface using vector store, we'll use it later to construct Q & A chain using LangChain.

In [ ]:
# Use similarity searching algorithm and return 3 most relevant documents.
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

**Now we have our vector store and retrieval system ready. We then need a large language model (LLM) to process information and answer the question.**

## Open-source LLMs from Hugging Face


### Hugging Face Local Pipelines

Hugging Face models can be run locally through the HuggingFacePipeline class.

- We need to install transformers python package.
- The Mistral-7B-v0.1 Large Language Model (LLM) is a pretrained generative text model with 7 billion parameters. Mistral-7B-v0.1 outperforms Llama-2-13B on all benchmarks tested. Read the [paper](https://arxiv.org/abs/2310.06825).
- Mistral-7B-v0.1's model size is 3.5GB, while Llama-2–13B has 13 billion parameters and 25GB model size.
- In order to use Llama2, you need to request access from Meta. Mistral-7B-v0.1 is publicly available already.

In [ ]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

hf = HuggingFacePipeline.from_model_id(
    model_id="mistralai/Mistral-7B-v0.1",
    task="text-generation",
    pipeline_kwargs={"temperature": 0, "max_new_tokens": 300}
)

llm = hf

In [ ]:
llm.invoke(query)

## Q & A chain 
Now we have both the retrieval system for relevant documents and LLM as QA chatbot ready.

We will take our initial query, together with the relevant documents retrieved based on the results of our similarity search, to create a prompt to feed into the LLM. The LLM will take the initial query as the question and relevant documents as the context information to generate a result.

Luckily, **LangChain** provides an abstraction of the whole pipeline - **RetrievalQA**

**Let's first construct a proper prompt for our task.**

Prompt engineering is another crucial factor in LLM's performance.

In [ ]:
prompt_template = """Use the following pieces of context to answer the question at the end. Please follow the following rules:
1. If you don't know the answer, don't try to make up an answer. Just say "I can't find the final answer but you may want to check the following links".
2. If you find the answer, write the answer in a concise way with five sentences maximum.

{context}

Question: {question}

Helpful Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

Call LangChain's RetrievalQA with the prompt above. 

In [ ]:
retrievalQA = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

## Use RetrievalQA invoke method to execute the chain
Note that Input of [invoke method](https://api.python.langchain.com/en/latest/chains/langchain.chains.retrieval_qa.base.RetrievalQA.html#langchain.chains.retrieval_qa.base.RetrievalQA.invoke) needs to be a dictionary.

In [ ]:
# Call the QA chain with our query.
result = retrievalQA.invoke({"query": query})

In [ ]:
print(result['result'])

In [ ]:
relevant_docs = result['source_documents']
print(f'There are {len(relevant_docs)} documents retrieved which are relevant to the query.')
print("*" * 100)
for i, doc in enumerate(relevant_docs):
    print(f"Relevant Document #{i+1}:\nSource file: {doc.metadata['source']}, Page: {doc.metadata['page']}\nContent: {doc.page_content}")
    print("-"*100)